# Fine-grained AST change types in C/C++ vulnerability fixes

In [ ]:
INSTALL = False

if INSTALL:
    %pip install -q -r requirements.txt

## Imports

In [ ]:
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from bisect import bisect_left
from collections import Counter
from importlib.metadata import version
from itertools import islice
from pathlib import Path

import ijson
import numpy as np
import pandas as pd
import tree_sitter_c
import tree_sitter_cpp
from joblib import Parallel, delayed
from scipy import stats
from tree_sitter import Language, Parser

sys.setrecursionlimit(20000)

## Configuration

In [ ]:
DATA = Path("data") / "megavul.json"
RESULTS = Path("results")
TABLES = RESULTS / "tables"
TOOLS = Path("tools")

SAMPLE = None
WORKERS = None
INVERSE = False
DEDUPE_CODE = True
LOADER = "v5"
HELD_OUT_SHARE = 0.2

DIFFERENCER = "gumtree"
GUMTREE_MATCHER = "gumtree-simple"
GUMTREE_MATCHERS = ["gumtree-classic", "gumtree-simple"]
GUMTREE_VERSION = "4.1.0"
GUMTREE_URL = (f"https://github.com/GumTreeDiff/gumtree/releases/download/v{GUMTREE_VERSION}"
               f"/gumtree-{GUMTREE_VERSION}.zip")
GUMTREE_JAR = TOOLS / f"gumtree-{GUMTREE_VERSION}" / "lib" / "gumtree.jar"
CLASSES = TOOLS / "classes"
JAVA, JAVAC = shutil.which("java"), shutil.which("javac")
CHUNK = 400
MATCH_MIN_HEIGHT = 2
MATCH_THRESHOLD = 0.5
MATCH_MAX_SIZE = 1000
MATCH_SIMILARITY = 0.3

LANGUAGES = {"c": tree_sitter_c.language, "cpp": tree_sitter_cpp.language}
GENERATORS = {"c": "c-treesitter-ng", "cpp": "cpp-treesitter-ng"}
EXTENSIONS = {".c": "c", ".cc": "cpp", ".cpp": "cpp", ".cxx": "cpp", ".hpp": "cpp"}
SEPARATORS = {",", ";", "(", ")", "{", "}"}

KEYS = ("cve_id", "cwe_ids", "repo_name", "commit_hash", "commit_date", "file_path", "func_name",
        "func_before", "func")
COLUMNS = ["cve", "cwe", "repo", "commit", "commit_date", "file", "function", "fold", "direction",
           "single_function", "kind", "type", "parent", "role", "feature", "before", "after", "size",
           "absorbed"]
FINE, COARSE = ["kind", "type", "parent"], ["kind", "type"]

DIFF_SAMPLE = 1000
TAU = 0.5
VALIDATION_N = 100

MIN_COMMITS = 10
COVERAGE_THRESHOLDS = (5, 10, 30)
EQUIVALENT = {"condition_clause": "parenthesized_expression"}
LANGUAGE_EFFECT = 0.02
MIN_CWE_COMMITS = 30
MIN_CWE_HITS = 5
CWE_EFFECT = 0.10
UNINFORMATIVE = {"CWE-Other", "NVD-CWE-Other", "NVD-CWE-noinfo", ""}
MIN_TYPE_COMMITS = 30
MIN_PAIR_COMMITS = 10
ALPHA = 0.05
TOP = 15

## Utilities

In [ ]:
class Node:
    def __init__(self, type, label, parent, role):
        self.type, self.label, self.parent, self.role = type, label, parent, role
        self.children = []

    def nodes(self):
        stack = [self]
        while stack:
            n = stack.pop()
            yield n
            stack.extend(reversed(n.children))

    def text(self, limit=80):
        return " ".join(self.src[self.start:self.end].decode(errors="replace").split())[:limit]

    def __repr__(self):
        return f"{self.type}({self.label})" if self.label else self.type


def parse(code, lang):
    src = code.encode()

    def convert(ts, role, parent):
        operator = not ts.is_named and role == "operator"
        n = Node("operator" if operator else ts.type,
                 "" if ts.child_count else ts.text.decode(errors="replace"), parent, role or "")
        n.token = not ts.is_named and not operator
        n.src, n.start, n.end = src, ts.start_byte, ts.end_byte
        n.children = [convert(c, ts.field_name_for_child(i), n)
                      for i, c in enumerate(ts.children) if c.type != "comment"]
        for i, child in enumerate(n.children):
            child.pos = i
        n.size = 1 + sum(child.size for child in n.children)
        n.height = 1 + max((child.height for child in n.children), default=0)
        n.key = hash((n.type, n.label, tuple(child.key for child in n.children)))
        return n

    tree = Parser(Language(LANGUAGES[lang]())).parse(src).root_node
    root = convert(tree, None, None)
    root.pos, root.has_error = 0, tree.has_error
    for i, n in enumerate(root.nodes()):
        n.pre, n.last = i, i + n.size - 1
    return root


def match(T1, T2):
    M, M2 = {}, {}

    def add(a, b):
        M[a] = b
        M2[b] = a

    def add_tree(a, b):
        pairs = list(zip(a.nodes(), b.nodes()))
        if all(x not in M and y not in M2 for x, y in pairs):
            for x, y in pairs:
                add(x, y)

    def dice(a, b):
        shared = sum(1 for d in islice(a.nodes(), 1, None) if d in M and b.pre <= M[d].pre <= b.last)
        return 2 * shared / (a.size + b.size - 2) if a.size + b.size > 2 else 0.0

    h1, h2 = {}, {}
    for n in T1.nodes():
        h1.setdefault(n.height, []).append(n)
    for n in T2.nodes():
        h2.setdefault(n.height, []).append(n)
    for h in range(max(h1.keys() | h2.keys()), MATCH_MIN_HEIGHT - 1, -1):
        index = {}
        for n2 in h2.get(h, []):
            index.setdefault(n2.key, []).append(n2)
        count1 = Counter(n.key for n in h1.get(h, []) if n not in M)
        ambiguous = []
        for n1 in h1.get(h, []):
            if n1 in M or n1.key not in index:
                continue
            candidates = [n2 for n2 in index[n1.key] if n2 not in M2]
            if len(candidates) == 1 and count1[n1.key] == 1:
                add_tree(n1, candidates[0])
            else:
                ambiguous += [(n1, n2) for n2 in candidates]
        ambiguous.sort(key=lambda p: (-(dice(p[0].parent, p[1].parent) if p[0].parent else 0),
                                      abs(p[0].pos - p[1].pos)))
        for n1, n2 in ambiguous:
            if n1 not in M and n2 not in M2:
                add_tree(n1, n2)

    def similar(a, b):
        la = Counter(n.label for n in a.nodes() if not n.children and not n.token)
        lb = Counter(n.label for n in b.nodes() if not n.children and not n.token)
        total = sum(la.values()) + sum(lb.values())
        return 2 * sum((la & lb).values()) / total if total else 0.0

    def recover(a, b):
        stack = [(a, b)]
        while stack:
            x, y = stack.pop()
            free1 = [k for k in x.children if k not in M]
            free2 = [k for k in y.children if k not in M2]
            for k1 in free1:
                k2 = next((k for k in free2 if k.key == k1.key), None)
                if k2 is not None and k1 not in M:
                    add_tree(k1, k2)
                    free2.remove(k2)
            free1 = [k for k in free1 if k not in M]
            for k1 in free1:
                same1 = [k for k in free1 if k.type == k1.type]
                same2 = [k for k in free2 if k.type == k1.type]
                if k1 is not same1[0] or not same2:
                    continue
                if len(same1) == len(same2) == 1 and (not k1.children or similar(k1, same2[0]) >= MATCH_SIMILARITY):
                    add(k1, same2[0])
                elif all(not k.children for k in same1 + same2):
                    for k, m in zip(same1, same2):
                        add(k, m)
            stack += [(k, M[k]) for k in x.children
                      if k in M and M[k].parent is y and k.key != M[k].key]
        if max(a.size, b.size) <= MATCH_MAX_SIZE:
            pool = {}
            for n in b.nodes():
                if not n.children and not n.token and n not in M2:
                    pool.setdefault((n.type, n.label), []).append(n)
            for n in a.nodes():
                options = pool.get((n.type, n.label))
                if n.children or n.token or n in M or not options:
                    continue
                options = [o for o in options if n.parent in M or o.parent in M2]
                if options:
                    best = next((o for o in options if o.parent.type == n.parent.type), options[0])
                    pool[(n.type, n.label)].remove(best)
                    add(n, best)

    for t1 in reversed(list(T1.nodes())):
        if t1 in M or not t1.children or t1.parent is None:
            continue
        shared = {}
        for d in islice(t1.nodes(), 1, None):
            a = M[d].parent if d in M else None
            while a is not None:
                if a not in M2 and a.type == t1.type:
                    shared[a] = shared.get(a, 0) + 1
                a = a.parent
        best = max(shared, key=lambda t2: (shared[t2] / (t1.size + t2.size - 2), -abs(t1.pos - t2.pos)),
                   default=None)
        if best is not None and 2 * shared[best] / (t1.size + best.size - 2) > MATCH_THRESHOLD:
            add(t1, best)
            recover(t1, best)

    if T1 not in M and T1.type == T2.type:
        add(T1, T2)
    recover(T1, T2)
    return M


def node(a):
    kind, n1, n2 = a
    return n2 if kind == "insert" else n1


def edit_script(T1, T2, M):
    M2 = {b: a for a, b in M.items()}
    A = [("delete", n1, None) for n1 in T1.nodes()
         if n1 not in M and n1.type not in SEPARATORS and (n1.parent is None or n1.parent in M)]
    A += [("insert", None, n2) for n2 in T2.nodes()
          if n2 not in M2 and n2.type not in SEPARATORS and (n2.parent is None or n2.parent in M2)]
    for n1, n2 in M.items():
        if n1.token:
            continue
        if n1.label != n2.label:
            A.append(("update", n1, n2))
        if n1.parent and M.get(n1.parent) is not n2.parent:
            A.append(("move", n1, n2))
    for p1, p2 in M.items():
        kept = [k for k in p1.children if not k.token and k in M and M[k].parent is p2]
        order, tails, ends, prev = [M[k].pos for k in kept], [], [], []
        for i, x in enumerate(order):
            j = bisect_left(tails, x)
            tails[j:j + 1], ends[j:j + 1] = [x], [i]
            prev.append(ends[j - 1] if j else None)
        stay, i = set(), ends[-1] if ends else None
        while i is not None:
            stay.add(i)
            i = prev[i]
        A += [("move", k, M[k]) for i, k in enumerate(kept) if i not in stay]
    return A


def offsets(code):
    src = code.encode()
    if len(src) == len(code):
        return lambda x: x
    table, b, u = [0] * (len(src) + 1), 0, 0
    for ch in code:
        nb, nu = len(ch.encode()), 2 if ord(ch) > 0xFFFF else 1
        for k in range(nb):
            table[b + k] = u
        b, u = b + nb, u + nu
    table[b] = u
    return table.__getitem__


def gumtree_nodes(texts, lang, matchers):
    with tempfile.TemporaryDirectory() as d:
        for pid, (before, after) in texts.items():
            Path(d, f"{pid}.a").write_bytes(before.encode())
            Path(d, f"{pid}.b").write_bytes(after.encode())
        out = Path(d, "matches.tsv")
        with open(out, "w", encoding="utf-8") as f:
            subprocess.run([JAVA, "-cp", os.pathsep.join([str(GUMTREE_JAR), str(CLASSES)]), "GumtreeBatch",
                            d, GENERATORS[lang], ",".join(matchers), "all"], stdout=f, check=True)
        result = {pid: {m: [] for m in matchers} for pid in texts}
        for line in open(out, encoding="utf-8"):
            p = line.rstrip("\n").split("\t")
            if p[0] == "M" and len(p) == 10:
                result[p[1]][p[2]].append((p[3], int(p[4]), int(p[5]), p[6], int(p[7]), int(p[8]), p[9]))
        return result


def gumtree_to_M(T1, T2, code1, code2, mapped):
    index = []
    for T, code in ((T1, code1), (T2, code2)):
        conv = offsets(code)
        index.append({})
        for n in T.nodes():
            name = (n.label if n.type == "operator" else n.type).replace("\t", "_").replace("\n", "_").replace("\r", "_")
            index[-1].setdefault((conv(n.start), conv(n.end), name), n)
    i1, i2 = index
    M, used, missing = {}, set(), 0
    for _, s0, s1, st, d0, d1, dt in mapped:
        a, b = i1.get((s0, s1, st)), i2.get((d0, d1, dt))
        if a is None or b is None:
            missing += 1
        elif a not in M and b not in used:
            M[a] = b
            used.add(b)
    for a in T1.nodes():
        if a.type == "operator" and a not in M and a.parent in M:
            b = next((k for k in M[a.parent].children
                      if k.type == "operator" and k not in used and k.role == a.role), None)
            if b is not None:
                M[a] = b
                used.add(b)
    return M, missing


def snippet(n):
    return "" if n is None else n.label or n.text()


def describe(a):
    kind, n1, n2 = a
    return f"{kind} {node(a).type}: {snippet(n1)!r} -> {snippet(n2)!r}"


def actions_of(r, lang, mapped=None):
    T1, T2 = parse(r["func_before"], lang), parse(r["func"], lang)
    if mapped:
        M, used = gumtree_to_M(T1, T2, r["func_before"], r["func"], mapped)[0], "gumtree"
    else:
        M, used = match(T1, T2), "ours"
    return T1, T2, M, edit_script(T1, T2, M), used


def rows(r, lang, mapped=None):
    T1, T2, M, A, used = actions_of(r, lang, mapped)
    direction = r.get("direction", "fix")
    record = (r["cve_id"], ";".join(r["cwe_ids"]), r["repo_name"], r["commit_hash"], r["commit_date"],
              r["file_path"], r["func_name"], r["fold"], direction, int(r["commit_functions"] == 1))
    matched = set(M.values())
    out = []
    for a in A:
        n = node(a)
        parent = n.parent.type if n.parent else ""
        absorbed = int(a[0] == "move" and (a[2].parent not in matched or a[1].parent not in M))
        out.append(record + (a[0], n.type, parent, n.role, f"{a[0].upper()}_{n.type}_{parent}",
                             snippet(a[1]), snippet(a[2]), n.size if a[0] != "update" else 1, absorbed))
    pair = {"commit": r["commit_hash"], "file": r["file_path"], "function": r["func_name"],
            "direction": direction, "actions": len(A), "parse_error": T1.has_error or T2.has_error,
            "nodes_before": T1.size, "nodes_after": T2.size, "matched": len(M), "differencer": used}
    return out, pair


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def save(df, name):
    df.to_csv(TABLES / f"{name}.csv")
    return df


def commit_presence(df):
    d = df.dropna(subset=["change_type"])
    return pd.crosstab(d["commit"], d["change_type"]).gt(0)


def derive_K(train, min_commits=MIN_COMMITS):
    fine = train.groupby(FINE)["commit"].nunique()
    coarse = train.groupby(COARSE)["commit"].nunique()
    return set(fine[fine >= min_commits].index), set(coarse[coarse >= min_commits].index)


def classify(df, K):
    fine, coarse = K
    out = []
    for kind, typ, parent in zip(df["kind"], df["type"], df["parent"]):
        if (kind, typ, parent) in fine:
            out.append(f"{kind.upper()}_{typ}_{parent}")
        elif (kind, typ) in coarse:
            out.append(f"{kind.upper()}_{typ}_*")
        else:
            out.append(None)
    return out


def statement_like(type):
    return type.endswith(("statement", "declaration", "definition")) or type == "else_clause"


def spans_of(T, code):
    conv = offsets(code)
    leaves, stmts, owner = {}, {}, {}
    stack = [(T, None)]
    while stack:
        n, enclosing = stack.pop()
        span = (conv(n.start), conv(n.end))
        if statement_like(n.type):
            stmts[span] = (n.label, n.type)
            enclosing = span
        if not n.children:
            leaves[span] = (n.label, n.type)
            owner[span] = enclosing
        stack.extend((k, enclosing) for k in reversed(n.children))
    return leaves, stmts, owner


def tokens_in(leaves, span):
    return Counter(label for (s, e), (label, _) in leaves.items()
                   if span[0] <= s and e <= span[1] and label not in SEPARATORS)


## Setup

In [ ]:
HAVE_GUMTREE = bool(JAVA and JAVAC)
if HAVE_GUMTREE:
    if not GUMTREE_JAR.exists():
        TOOLS.mkdir(exist_ok=True)
        archive = TOOLS / "gumtree.zip"
        urllib.request.urlretrieve(GUMTREE_URL, archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(TOOLS)
        archive.unlink()
    CLASSES.mkdir(exist_ok=True)
    subprocess.run([JAVAC, "-cp", str(GUMTREE_JAR), "-d", str(CLASSES), str(TOOLS / "GumtreeBatch.java")],
                   check=True)
if not HAVE_GUMTREE:
    DIFFERENCER = "ours"

In [ ]:
RESULTS.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)
SHA = sha256(DATA)
CACHE = DATA.with_name(f"megavul_pairs_{SHA[:16]}_{LOADER}.json")

if CACHE.exists():
    saved = json.loads(CACHE.read_text(encoding="utf-8"))
    pairs, dropped = saved["pairs"], Counter(saved["dropped"])
else:
    vulnerable, total = [], 0
    with open(DATA, "rb") as f:
        for r in ijson.items(f, "item"):
            total += 1
            if r.get("is_vul") and r.get("func_before"):
                vulnerable.append({k: r.get(k) for k in KEYS})

    dropped = Counter({"not a vulnerable function": total - len(vulnerable)})
    by_commit, by_code = {}, set()
    for r in vulnerable:
        lang = EXTENSIONS.get("." + r["file_path"].rsplit(".", 1)[-1].lower())
        if r["func_before"] == r["func"]:
            dropped["function unchanged"] += 1
        elif lang is None:
            dropped["header or unknown extension"] += 1
        else:
            key = (r["commit_hash"], r["file_path"], r["func_name"], r["func_before"], r["func"])
            if key in by_commit:
                dropped["same function and commit under another CVE record"] += 1
                by_commit[key][1]["cves"].add(r["cve_id"])
                by_commit[key][1]["cwes"].update(r["cwe_ids"] or [])
            else:
                by_commit[key] = (lang, {**r, "cves": {r["cve_id"]}, "cwes": set(r["cwe_ids"] or [])})

    pairs = {"c": [], "cpp": []}
    for lang, r in by_commit.values():
        if DEDUPE_CODE and (r["func_before"], r["func"]) in by_code:
            dropped["identical function pair in another commit"] += 1
            continue
        by_code.add((r["func_before"], r["func"]))
        pairs[lang].append(r)

    per_commit = Counter(r["commit_hash"] for p in pairs.values() for r in p)
    commits = {}
    for p in pairs.values():
        for r in p:
            commits.setdefault(r["repo_name"], set()).add(r["commit_hash"])
    sizes = sorted(((len(v), k) for k, v in commits.items()), key=lambda t: (-t[0], t[1]))
    target, used, folds = {"train": 1 - HELD_OUT_SHARE, "test": HELD_OUT_SHARE}, {"train": 0, "test": 0}, {}
    for n, repo in sizes:
        fold = min(("train", "test"), key=lambda f: used[f] / target[f])
        folds[repo], used[fold] = fold, used[fold] + n

    for p in pairs.values():
        for r in p:
            r["cve_id"], r["cwe_ids"] = ";".join(sorted(r.pop("cves"))), sorted(r.pop("cwes"))
            r["commit_functions"] = per_commit[r["commit_hash"]]
            r["fold"] = folds[r["repo_name"]]
    CACHE.write_text(json.dumps({"pairs": pairs, "dropped": dropped}), encoding="utf-8")

if SAMPLE:
    pairs = {lang: random.Random(0).sample(p, min(SAMPLE, len(p))) for lang, p in pairs.items()}

environment = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "packages": {p: version(p) for p in ("tree-sitter", "tree-sitter-c", "tree-sitter-cpp", "pandas", "numpy",
                                          "scipy", "ijson", "joblib")},
    "differencer": DIFFERENCER, "megavul_sha256": SHA, "megavul_megabytes": round(DATA.stat().st_size / 1e6, 1)}
if HAVE_GUMTREE:
    environment |= {"gumtree": GUMTREE_VERSION, "gumtree_matcher": GUMTREE_MATCHER,
                    "gumtree_parameters": "defaults of the release, no overrides",
                    "gumtree_jar_sha256": sha256(GUMTREE_JAR),
                    "java": subprocess.run([JAVA, "-version"], capture_output=True, text=True).stderr.splitlines()[0]}
(RESULTS / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

filters = pd.Series(dropped, name="records")
filters["kept: C function pairs"] = len(pairs["c"])
filters["kept: C++ function pairs"] = len(pairs["cpp"])
display(save(filters.to_frame(), "filters"))

dataset = {}
for lang, p in pairs.items():
    per_commit = Counter(r["commit_hash"] for r in p)
    held_out = [r for r in p if r["fold"] == "test"]
    dataset[lang] = {
        "function pairs": len(p), "commits": len(per_commit), "repositories": len({r["repo_name"] for r in p}),
        "CVEs": len({c for r in p for c in r["cve_id"].split(";")}),
        "CWE types": len({c for r in p for c in r["cwe_ids"]}),
        "median functions per commit": float(np.median(list(per_commit.values()))),
        "90th percentile": float(np.percentile(list(per_commit.values()), 90)),
        "maximum": max(per_commit.values()),
        "single-function commits": round(float(np.mean([n == 1 for n in per_commit.values()])), 3),
        "held-out commits": len({r["commit_hash"] for r in held_out}),
        "held-out repositories": len({r["repo_name"] for r in held_out})}
display(save(pd.DataFrame(dataset), "dataset"))

top_repositories = pd.concat({lang: pd.Series(Counter(r["repo_name"] for r in {r["commit_hash"]: r for r in p}.values()))
                              .sort_values(ascending=False).head(TOP) for lang, p in pairs.items()},
                             names=["language", "repository"]).rename("commits")
display(save(top_repositories.to_frame(), "top_repositories"))

In [ ]:
BEFORE = "int f(int len, int max) { if (len > max) return -1; return 0; }"
AFTER = "int f(int len, int max) { if (len >= max) return -1; return 0; }"
for differencer in ("ours", "gumtree") if HAVE_GUMTREE else ("ours",):
    mapped = None
    if differencer == "gumtree":
        mapped = gumtree_nodes({"x": (BEFORE, AFTER)}, "c", [GUMTREE_MATCHER])["x"][GUMTREE_MATCHER]
    A = actions_of({"func_before": BEFORE, "func": AFTER}, "c", mapped)[3]
    assert len(A) == 1 and A[0][0] == "update" and A[0][2].label == ">="

In [ ]:
actions, records = {}, {}
for lang in ("c", "cpp"):
    subsets = [pairs[lang]] + ([[{**r, "func_before": r["func"], "func": r["func_before"], "direction": "introduction"}
                                for r in pairs[lang]]] if INVERSE else [])
    results = []
    with Parallel(n_jobs=WORKERS or -1, batch_size=16) as parallel:
        for subset in subsets:
            if DIFFERENCER == "gumtree":
                for start in range(0, len(subset), CHUNK):
                    chunk = subset[start:start + CHUNK]
                    texts = {f"{i:04d}": (r["func_before"], r["func"]) for i, r in enumerate(chunk)}
                    mapped = gumtree_nodes(texts, lang, [GUMTREE_MATCHER])
                    results += parallel(delayed(rows)(r, lang, mapped[f"{i:04d}"][GUMTREE_MATCHER])
                                        for i, r in enumerate(chunk))
            else:
                results += parallel(delayed(rows)(r, lang) for r in subset)
    actions[lang] = pd.DataFrame([row for rs, _ in results for row in rs], columns=COLUMNS)
    records[lang] = pd.DataFrame([pair for _, pair in results])
    actions[lang].to_csv(RESULTS / f"actions_{lang}.csv", index=False)
    records[lang].to_csv(RESULTS / f"pairs_{lang}.csv", index=False)

pipeline = {}
for lang in actions:
    fixes = records[lang][records[lang]["direction"] == "fix"]
    a = actions[lang][actions[lang]["direction"] == "fix"]
    pipeline[lang] = {
        "function pairs": len(fixes),
        "matched by GumTree": int((fixes["differencer"] == "gumtree").sum()),
        "no edit actions": int((fixes["actions"] == 0).sum()),
        "function pairs analysed": int((fixes["actions"] > 0).sum()),
        "with parse errors": int(fixes["parse_error"].sum()),
        "proportion without parse errors": round(1 - fixes["parse_error"].mean(), 3),
        "edit actions": len(a), "moves": int((a["kind"] == "move").sum()),
        "absorbed moves": int(a["absorbed"].sum()),
        "analysed actions": int((a["absorbed"] == 0).sum()),
        "analysed actions of type ERROR": int(((a["absorbed"] == 0) & (a["type"] == "ERROR")).sum())}
display(save(pd.DataFrame(pipeline), "pipeline"))

kinds = pd.DataFrame({lang: a[(a["direction"] == "fix") & (a["absorbed"] == 0)]["kind"].value_counts(normalize=True)
                      for lang, a in actions.items()})
display(save(kinds.round(3), "action_kinds"))

In [ ]:
features = {}
for lang, df in actions.items():
    df = df[df["absorbed"] == 0]
    present = df.assign(present=1).pivot_table(index=["commit", "direction"], columns="feature",
                                               values="present", aggfunc="max", fill_value=0)
    kept = present.loc[:, present.sum() >= MIN_COMMITS]
    meta = df.drop_duplicates(["commit", "direction"]).set_index(["commit", "direction"])[["repo", "fold"]]
    features[lang] = meta.join(kept, how="right").reset_index()
    features[lang].to_csv(RESULTS / f"features_{lang}.csv", index=False)

display(save(pd.DataFrame({lang: {
    "rows (commits and directions)": len(f), "change types before the support filter": int(actions[lang]["feature"].nunique()),
    f"change types with support >= {MIN_COMMITS}": f.shape[1] - 4, "repositories": int(f["repo"].nunique()),
    "rows in the held-out fold": int((f["fold"] == "test").sum())} for lang, f in features.items()}), "features"))

changes, K = {}, {}
for lang, df in actions.items():
    d = df[(df["direction"] == "fix") & (df["absorbed"] == 0)].copy()
    K[lang] = derive_K(d[d["fold"] == "train"])
    d["change_type"] = classify(d, K[lang])
    changes[lang] = d

In [ ]:
CODES = {"insert": "INS", "delete": "DEL", "update": "UPD", "move": "MOV"}
for lang, df in actions.items():
    df = df[df["absorbed"] == 0]
    with open(RESULTS / f"change_frequency_{lang}.jsonl", "w", encoding="utf-8") as f:
        for (commit, direction), g in df.groupby(["commit", "direction"], sort=True):
            entity = g["type"].value_counts()
            parent = (g["kind"].map(CODES) + "_" + g["type"] + "_" + g["parent"]).value_counts()
            f.write(json.dumps({
                "commit": commit, "repo": g["repo"].iloc[0], "direction": direction,
                "frequency": [{"c": c, "f": str(n)} for c, n in entity.items()],
                "frequencyParent": [{"c": c, "f": str(n)} for c, n in parent.items()]}) + "\n")

## Differencer validation

In [ ]:
validation = {}
for lang in ("c", "cpp"):
    path = RESULTS / f"validation_{lang}.csv"
    if not path.exists():
        sample = random.Random(0).sample(pairs[lang], min(VALIDATION_N, len(pairs[lang])))
        mapped = {}
        if DIFFERENCER == "gumtree":
            mapped = gumtree_nodes({f"{i:03d}": (r["func_before"], r["func"]) for i, r in enumerate(sample)},
                                   lang, [GUMTREE_MATCHER])
        out = []
        for i, r in enumerate(sample):
            T1, T2, M, A, used = actions_of(r, lang, mapped.get(f"{i:03d}", {}).get(GUMTREE_MATCHER))
            out.append({"commit": r["commit_hash"], "file": r["file_path"], "function": r["func_name"],
                        "before": r["func_before"], "after": r["func"],
                        "script": "\n".join(describe(a) for a in A), "differencer": used,
                        "mapping_correct": "", "notes": ""})
        pd.DataFrame(out).to_csv(path, index=False)
    validation[lang] = pd.read_csv(path, keep_default_na=False)

accuracy = {}
for lang, v in validation.items():
    yes, no = int((v["mapping_correct"] == "yes").sum()), int((v["mapping_correct"] == "no").sum())
    low, high = stats.binomtest(yes, yes + no).proportion_ci(method="wilson") if yes + no else (np.nan, np.nan)
    accuracy[lang] = {"pairs": len(v), "labelled yes": yes, "labelled no": no,
                      "proportion accurate": yes / (yes + no) if yes + no else np.nan,
                      "95% interval, low": low, "95% interval, high": high}
display(save(pd.DataFrame(accuracy).round(3), "validation_labels"))


In [ ]:
diff_rows = []
for lang in ("c", "cpp"):
    labels = {(r.commit, r.file, r.function): r.mapping_correct for r in validation[lang].itertuples()}
    index = {(r["commit_hash"], r["file_path"], r["func_name"]): i for i, r in enumerate(pairs[lang])}
    chosen = {index[k] for k in labels if k in index}
    pool = list(range(len(pairs[lang])))
    chosen |= set(random.Random(1).sample(pool, min(DIFF_SAMPLE, len(pool)))) if DIFF_SAMPLE else set(pool)
    texts = {f"{i:06d}": (pairs[lang][i]["func_before"], pairs[lang][i]["func"]) for i in sorted(chosen)}
    theirs = gumtree_nodes(texts, lang, GUMTREE_MATCHERS)
    for pid, (before, after) in texts.items():
        r = pairs[lang][int(pid)]
        T1, T2 = parse(before, lang), parse(after, lang)
        c1, c2 = offsets(before), offsets(after)
        maps = {"ours": {"leaf": {}, "stmt": {}}}
        for m in GUMTREE_MATCHERS:
            maps[m] = {"leaf": {}, "stmt": {}}
            for kind, s0, s1, _, d0, d1, _ in theirs[pid][m]:
                if kind in ("L", "S"):
                    maps[m]["leaf" if kind == "L" else "stmt"].setdefault((s0, s1), (d0, d1))
        for a, b in match(T1, T2).items():
            kind = "stmt" if statement_like(a.type) else "leaf" if not a.children else None
            if kind:
                maps["ours"][kind].setdefault((c1(a.start), c1(a.end)), (c2(b.start), c2(b.end)))
        leaves1, stmts1, owner1 = spans_of(T1, before)
        leaves2, stmts2, _ = spans_of(T2, after)
        names = list(maps)
        flags = {k: {"statements": 0, "tokens": 0} for k in names}
        agreed = {}
        for span in stmts1:
            partners = {k: maps[k]["stmt"].get(span) for k in names}
            if all(p is None for p in partners.values()):
                continue
            agreed[span] = len(set(partners.values())) == 1
            if agreed[span]:
                continue
            source = tokens_in(leaves1, span)
            sims = {}
            for k, p in partners.items():
                target = tokens_in(leaves2, p) if p else Counter()
                total = sum(source.values()) + sum(target.values())
                sims[k] = 2 * sum((source & target).values()) / total if p and total else 0.0
            best = max(sims.values())
            for k in names:
                if (partners[k] is None and best >= TAU) or (partners[k] is not None and sims[k] < best):
                    flags[k]["statements"] += 1
        for span, (label, type_) in leaves1.items():
            if label in SEPARATORS or not agreed.get(owner1[span], False):
                continue
            partners = {k: maps[k]["leaf"].get(span) for k in names}
            if type_ == "operator":
                partners = {k: p for k, p in partners.items() if p is not None}
            if len(partners) < 2 or len(set(partners.values())) == 1:
                continue
            if any(p is not None and p not in leaves2 for p in partners.values()):
                continue
            score = {k: int(p is not None and leaves2[p][0] == label) for k, p in partners.items()}
            best = max(score.values())
            for k in score:
                if score[k] < best:
                    flags[k]["tokens"] += 1
        label = labels.get((r["commit_hash"], r["file_path"], r["func_name"]), "")
        diff_rows += [{"lang": lang, "pair": int(pid), "algorithm": k, "label": label,
                       "flagged": bool(v["statements"] or v["tokens"])} for k, v in flags.items()]

differential = pd.DataFrame(diff_rows)
differential.to_csv(RESULTS / "differential.csv", index=False)
rate = differential.groupby(["lang", "algorithm"])["flagged"].agg(["mean", "sum", "count"])
rate.columns = ["proportion of pairs with an inaccurate mapping", "pairs flagged", "pairs"]
display(save(rate.round(3), "validation_differential"))

detector = {}
for lang, g in differential[(differential["algorithm"] == GUMTREE_MATCHER) & (differential["label"] != "")].groupby("lang"):
    tp = int((g["flagged"] & (g["label"] == "no")).sum())
    fp = int((g["flagged"] & (g["label"] == "yes")).sum())
    fn = int((~g["flagged"] & (g["label"] == "no")).sum())
    detector[lang] = {"labelled pairs": len(g), "labelled no": tp + fn, "flagged": tp + fp,
                      "precision": round(tp / max(tp + fp, 1), 2), "recall": round(tp / max(tp + fn, 1), 2)}
display(save(pd.DataFrame(detector), "validation_detector"))

In [ ]:
equivalence = {}
for lang in ("c", "cpp"):
    v = validation[lang]
    theirs = gumtree_nodes({f"{i:03d}": (r.before, r.after) for i, r in v.iterrows()}, lang, [GUMTREE_MATCHER])
    same = common = n_ours = n_theirs = missing = mapped = 0
    for i, r in v.iterrows():
        T1, T2 = parse(r.before, lang), parse(r.after, lang)
        rows_i = theirs[f"{i:03d}"][GUMTREE_MATCHER]
        M, miss = gumtree_to_M(T1, T2, r.before, r.after, rows_i)
        ours = Counter(describe(a) for a in edit_script(T1, T2, match(T1, T2)))
        gumtree = Counter(describe(a) for a in edit_script(T1, T2, M))
        same += ours == gumtree
        common += sum((ours & gumtree).values())
        n_ours, n_theirs = n_ours + sum(ours.values()), n_theirs + sum(gumtree.values())
        missing, mapped = missing + miss, mapped + len(rows_i)
    equivalence[lang] = {"pairs": len(v), "identical edit script": round(same / len(v), 3),
                         "actions: own matcher": n_ours, "actions: GumTree": n_theirs,
                         "actions in common, share of own matcher's": round(common / n_ours, 3),
                         "GumTree nodes not found in our trees": round(missing / mapped, 4)}
display(save(pd.DataFrame(equivalence), "validation_equivalence"))

## RQ1: Completeness

In [ ]:
coverage = {}
for lang, d in changes.items():
    train, test = d[d["fold"] == "train"], d[d["fold"] == "test"]
    for threshold in COVERAGE_THRESHOLDS:
        k = derive_K(train, threshold)
        coverage[(lang, threshold)] = {
            "change types": len(k[0]), "fallback types": len(k[1]),
            "actions, training fold": len(train), "actions, held-out fold": len(test),
            "coverage, training fold": pd.Series(classify(train, k)).notna().mean(),
            "coverage, held-out fold, fine types only": pd.Series(classify(test, (k[0], set()))).notna().mean(),
            "coverage, held-out fold": pd.Series(classify(test, k)).notna().mean()}
coverage = pd.DataFrame(coverage).T
coverage.index.names = ["language", "minimum commits"]
display(save(coverage.round(4), "rq1_coverage"))

unclassified = pd.concat({lang: d[(d["fold"] == "test") & d["change_type"].isna()].value_counts(FINE).head(TOP)
                          for lang, d in changes.items()}, names=["language"]).rename("actions")
display(save(unclassified.to_frame(), "rq1_unclassified"))

for lang, d in changes.items():
    taxonomy = d.dropna(subset=["change_type"]).groupby("change_type").agg(
        actions=("commit", "size"), commits=("commit", "nunique"), repositories=("repo", "nunique"))
    save(taxonomy.sort_values("commits", ascending=False), f"rq1_taxonomy_{lang}")

## RQ2: Distribution

In [ ]:
variants = ["all fixes", "single-function commits", "no parse error"]
robustness, concentration, examples = {}, {}, []
for lang, d in changes.items():
    flags = records[lang][records[lang]["direction"] == "fix"].drop_duplicates(["commit", "file", "function"])
    parsed = d.merge(flags.loc[~flags["parse_error"], ["commit", "file", "function"]], on=["commit", "file", "function"])
    subsets = {"all fixes": d, "single-function commits": d[d["single_function"] == 1], "no parse error": parsed}
    shares = {}
    for name, v in subsets.items():
        v = v.dropna(subset=["change_type"])
        shares[name] = commit_presence(v).mean()
        shares[name + ": share of actions"] = v["change_type"].value_counts(normalize=True)
    table = pd.DataFrame(shares).fillna(0).sort_values("all fixes", ascending=False)
    robustness[lang] = {name: stats.spearmanr(table["all fixes"], table[name]).statistic for name in variants[1:]}
    by_actions = table["all fixes: share of actions"].sort_values(ascending=False)
    concentration[lang] = {"change types": len(table), "commits": d["commit"].nunique(),
                           "types making up half of all actions": int((by_actions.cumsum() < 0.5).sum() + 1)}
    for t in table.index[:TOP]:
        found = d[d["change_type"] == t]
        examples.append(found.sample(min(2, len(found)), random_state=0).assign(language=lang)[
            ["language", "change_type", "repo", "commit", "file", "function", "before", "after"]])
    save(table, f"rq2_distribution_{lang}")
    display(table.head(TOP).round(3))

display(save(pd.DataFrame(concentration), "rq2_concentration"))
display(save(pd.DataFrame(robustness).round(3), "rq2_robustness"))
save(pd.concat(examples).set_index(["language", "change_type"]), "rq2_examples")

## RQ3: Languages

In [ ]:
grammar = {lang: Language(LANGUAGES[lang]()) for lang in LANGUAGES}
kinds = {lang: {g.node_kind_for_id(i) for i in range(g.node_kind_count) if g.node_kind_is_named(i)}
         for lang, g in grammar.items()}
cpp_only = kinds["cpp"] - kinds["c"] - set(EQUIVALENT)

display(save(pd.DataFrame({lang: {
    "actions": len(d),
    "actions touching a C++-only node type": int((d["type"].isin(cpp_only) | d["parent"].isin(cpp_only)).sum()),
    "proportion of actions": (d["type"].isin(cpp_only) | d["parent"].isin(cpp_only)).mean(),
    "proportion of commits with such an action": d.loc[d["type"].isin(cpp_only) | d["parent"].isin(cpp_only),
                                                       "commit"].nunique() / d["commit"].nunique()}
    for lang, d in changes.items()}).round(4), "rq3_cpp_only"))

presence = {}
for lang, d in changes.items():
    d = d.dropna(subset=["change_type"]).copy()
    for old, new in EQUIVALENT.items():
        d["change_type"] = d["change_type"].str.replace(old, new, regex=False)
    presence[lang] = commit_presence(d)

touches = (changes["cpp"].dropna(subset=["change_type"])
           .assign(touches=lambda x: x["type"].isin(cpp_only) | x["parent"].isin(cpp_only))
           .groupby("change_type")["touches"].any())
for old, new in EQUIVALENT.items():
    touches.index = touches.index.str.replace(old, new, regex=False)
names = {lang: set(p.columns) for lang, p in presence.items()}
display(save(pd.Series({
    "change types in both languages": len(names["c"] & names["cpp"]),
    "change types only in C": len(names["c"] - names["cpp"]),
    "change types only in C++": len(names["cpp"] - names["c"]),
    "change types only in C++, with a C++-only node type": int(touches.reindex(sorted(names["cpp"] - names["c"])).fillna(False).sum())},
    name="change types").to_frame(), "rq3_taxonomy_overlap"))

found = []
for t in sorted(set(presence["c"].columns) | set(presence["cpp"].columns)):
    n_c = int(presence["c"][t].sum()) if t in presence["c"] else 0
    n_p = int(presence["cpp"][t].sum()) if t in presence["cpp"] else 0
    if max(n_c, n_p) < MIN_COMMITS:
        continue
    N_c, N_p = len(presence["c"]), len(presence["cpp"])
    _, p = stats.fisher_exact([[n_c, N_c - n_c], [n_p, N_p - n_p]])
    found.append({"change type": t, "C": n_c / N_c, "C++": n_p / N_p, "difference (C++ - C)": n_p / N_p - n_c / N_c,
                  "commits in C": n_c, "commits in C++": n_p, "p": p})
differences = pd.DataFrame(found)
differences["q"] = stats.false_discovery_control(differences["p"])
differences = differences.sort_values("difference (C++ - C)", key=abs, ascending=False)
significant = differences[differences["q"] < ALPHA]
large = significant[significant["difference (C++ - C)"].abs() >= LANGUAGE_EFFECT]
save(differences.set_index("change type"), "rq3_differences")
display(save(pd.DataFrame({"change types compared": [len(differences)], "differ at q < 0.05": [len(significant)],
                           "and by at least 2 points of commits": [len(large)]}), "rq3_summary"))
display(save(large.set_index("change type").head(TOP).round(4), "rq3_large_differences"))

## RQ4: Weakness types

In [ ]:
summary = {}
for lang, d in changes.items():
    presence = commit_presence(d)
    presence = presence.loc[:, presence.sum() >= MIN_COMMITS]
    per_commit = d.groupby("commit").agg(cwe=("cwe", lambda s: set(";".join(s).split(";")) - UNINFORMATIVE),
                                         repo=("repo", "first")).loc[presence.index]
    counts = pd.Series([c for s in per_commit["cwe"] for c in s]).value_counts()
    cwes = counts[counts >= MIN_CWE_COMMITS].index
    found = []
    for cwe in cwes:
        has = per_commit["cwe"].map(lambda s: cwe in s).to_numpy()
        for t in presence.columns:
            x = presence[t].to_numpy()
            n11, n10, n01, n00 = int((x & has).sum()), int((~x & has).sum()), int((x & ~has).sum()), int((~x & ~has).sum())
            if n11 < MIN_CWE_HITS:
                continue
            _, p = stats.fisher_exact([[n11, n10], [n01, n00]], alternative="greater")
            top = per_commit.loc[x & has, "repo"].value_counts(normalize=True)
            found.append({"CWE": cwe, "change type": t, "commits of the CWE": n11 + n10, "with the type": n11,
                          "proportion in the CWE": n11 / (n11 + n10), "proportion elsewhere": n01 / (n01 + n00),
                          "p": p, "top project share": top.iloc[0]})
    tests = pd.DataFrame(found)
    tests["difference"] = tests["proportion in the CWE"] - tests["proportion elsewhere"]
    tests["q"] = stats.false_discovery_control(tests["p"])
    characteristic = tests[(tests["q"] < ALPHA) & (tests["difference"] >= CWE_EFFECT)]
    reported = characteristic[characteristic["top project share"] <= 0.5].sort_values(["q", "difference"], ascending=[True, False])
    summary[lang] = {"CWEs tested": len(cwes), "tests": len(tests), "characteristic pairs": len(characteristic),
                     "dominated by one project": len(characteristic) - len(reported), "reported": len(reported)}
    examples = []
    for (cwe, t), _ in reported.set_index(["CWE", "change type"]).head(5).iterrows():
        ids = per_commit.index[per_commit["cwe"].map(lambda s: cwe in s)]
        found_ = d[(d["change_type"] == t) & d["commit"].isin(ids)]
        examples.append(found_.sample(1, random_state=0).assign(CWE=cwe)[
            ["CWE", "change_type", "repo", "commit", "file", "function", "before", "after"]])
    if examples:
        save(pd.concat(examples).set_index(["CWE", "change_type"]), f"rq4_examples_{lang}")
    save(counts[cwes].rename("commits").to_frame(), f"rq4_cwe_counts_{lang}")
    save(tests.set_index(["CWE", "change type"]), f"rq4_tests_{lang}")
    display(save(reported.set_index(["CWE", "change type"]).round(4), f"rq4_reported_{lang}"))

display(save(pd.DataFrame(summary), "rq4_summary"))

## RQ5: Co-occurrence

In [ ]:
summary = {}
for lang, d in changes.items():
    multi = d.groupby("commit")["change_type"].transform("nunique") >= 2
    results, sizes = {}, {}
    for name, subset in (("all", d[multi]), ("single-function", d[multi & (d["single_function"] == 1)])):
        presence = commit_presence(subset)
        presence = presence.loc[:, presence.sum() >= MIN_TYPE_COMMITS]
        X = presence.to_numpy().astype(int)
        N, together, n = len(X), X.T @ X, X.sum(axis=0)
        found = []
        for i in range(len(n)):
            for j in range(i + 1, len(n)):
                a = together[i, j]
                if a < MIN_PAIR_COMMITS:
                    continue
                _, p = stats.fisher_exact([[a, n[i] - a], [n[j] - a, N - n[i] - n[j] + a]], alternative="greater")
                found.append({"type A": presence.columns[i], "type B": presence.columns[j],
                              "commits with both": int(a), "lift": a * N / (n[i] * n[j]), "p": p})
        results[name] = pd.DataFrame(found)
        results[name]["q"] = stats.false_discovery_control(results[name]["p"])
        sizes[name] = N
    above = results["all"][(results["all"]["q"] < ALPHA) & (results["all"]["lift"] > 1)]
    robust = above.merge(results["single-function"][results["single-function"]["q"] < ALPHA][["type A", "type B", "lift"]],
                         on=["type A", "type B"], suffixes=("", ": single-function")).sort_values("lift", ascending=False)
    summary[lang] = {"commits with at least two change types": sizes["all"],
                     "of which single-function commits": sizes["single-function"],
                     "type pairs tested": len(results["all"]), "co-occurring more than expected": len(above),
                     "and also in single-function commits": len(robust)}
    examples = []
    for _, pair in robust.head(5).iterrows():
        both = d.groupby("commit")["change_type"].agg(set).map(lambda s: {pair["type A"], pair["type B"]} <= s)
        commit = both[both].index.sort_values()[0]
        examples.append(d[(d["commit"] == commit) & d["change_type"].isin([pair["type A"], pair["type B"]])].assign(
            pair=f"{pair['type A']} + {pair['type B']}")[["pair", "repo", "commit", "file", "function", "change_type",
                                                          "before", "after"]])
    if examples:
        save(pd.concat(examples).set_index(["pair", "commit"]), f"rq5_examples_{lang}")
    save(results["all"].set_index(["type A", "type B"]), f"rq5_pairs_{lang}")
    display(save(robust.set_index(["type A", "type B"]).head(TOP).round(3), f"rq5_robust_{lang}"))

display(save(pd.DataFrame(summary), "rq5_summary"))